# Getting familiar with UCLCHEM

> *For some of the problems one or multiple hints are given in italics. Try solving the problems first without the hints!*

## 0  Installation

Download the UCLCHEM network from GitHub and install it.  
[Full installation documentation](https://uclchem.github.io/v3.5.5/getting-started/installation.html)

```bash
git clone https://github.com/uclchem/UCLCHEM.git
cd UCLCHEM
git switch develop
pip install .
```

Check that the install succeeded by looking for a line like  
`Successfully installed uclchem-#VERSION` in the output.

## 1  Inspecting the Input Files
`species.csv` and `reactions.csv` contain a human (and machine) readable version of all 
the reactions that are compiled into the uclchem reaction network. They are copied 
to the python source directory whenever you run Makerates. 



### `src/uclchem/species.csv`

Open the species list with your preferred editor (we choose nano here), e.g.:

```bash
nano src/uclchem/species.csv &
```

Each row describes one chemical species; the columns are listed below. The first 3 numbers are most important!

| # | Column | Description |
|---|--------|-------------|
| 1 | `NAME` | Species name |
| 2 | `MASS` | Species mass (amu) |
| 3 | `BINDING_ENERGY` | Binding energy to grain surface (K) |
| 4 | `SOLID_FRACTION` | Fraction of desorption into solid phase |
| 5 | `MONO_FRACTION` | Fraction of desorption into monolayer |
| 6 | `VOLCANO_FRACTION` | Fraction released via volcano desorption |
| 7 | `ENTHALPY` | Formation enthalpy (kJ/mol) |
| 8 | `DESORPTION_PREF` | Pre-exponential factor for thermal desorption |
| 9 | `DIFFUSION_BARRIER` | Barrier for surface diffusion (K) |
| 10 | `DIFFUSION_PREF` | Pre-exponential factor for diffusion |
| 11–13 | `IX`, `IY`, `IZ` | Moments of inertia (for partition functions) |
| 14 | `SYMMETRY_NUMBER` | Rotational symmetry number |

Species appear **three times** in different phases.  For example `H2O` is gas-phase water,  
`#H2O` is surface-ice water (on grain surfaces), and `@H2O` is bulk-ice water (inside the mantle).

### `src/uclchem/reactions.csv`

```bash
nano src/uclchem/reactions.csv &
```

| Columns | Column names | Content |
|---------|-------------|---------|
| 1–3 | `Reactant 1–3` | Reactants |
| 4–7 | `Product 1–4` | Products |
| 8–10 | `Alpha`, `Beta`, `Gamma` | Rate parameters α, β, γ |
| 11–12 | `T_min`, `T_max` | Valid temperature range (K) |
| 13 | `reduced_mass` | Reduced mass of reactants |
| 14 | `extrapolate` | Whether to extrapolate outside temperature range |
| 15 | `exothermicity` | Reaction exothermicity (erg per reaction) |

Some special pseudo-reactant labels:  
- `FREEZE` → freeze-out reaction (gas species sticks to grain)  
- `BULKSWAP` → exchange between bulk ice and surface ice layer
- `THERM` → thermal desorption

### `src/fortran_src/chemistry.f90`

```bash
nano src/fortran_src/chemistry.f90 &
```

Find the subroutine `integrateODESystem`.  It is called at every timestep (via `solveAbundances` in `wrap.f90`) and drives the ODE solver `DVODE_F90`, which receives:
- `F` — the chemical network as a system of ODEs  
- `NEQ` — number of equations  
- `abund` — current species abundances  
- `currentTime`, `targetTime` — time integration interval

## Exercise (a.1) — Reaction Types
Scroll through `reactions.csv` and find **one example** of each reaction type below:

- Radiative association
- Dissociative recombination
- Photodissociation
- Charge transfer
- Ion–molecule reaction
- Neutral–neutral reaction



### ✏️ Your answer (a.1)

*Double-click this cell to edit.*

- **Radiative association:** `...`
- **Dissociative recombination:** `...`
- **Photodissociation:** `...`
- **Charge transfer:** `...`
- **Ion–molecule reaction:** `...`
- **Neutral–neutral reaction:** `...`

<details>
<summary><b>Solution (a) — click to expand</b></summary>

| Type | Example |
|------|---------|
| Radiative association | H + OH → H₂O + hν |
| Dissociative recombination | HCO⁺ + e⁻ → CO + H |
| Photodissociation | H₂O + hν → OH + H |
| Charge transfer | OH⁺ + H₂O → OH + H₂O⁺ |
| Ion–molecule reaction | HCO⁺ + H₂O → CO + H₃O⁺ |
| Neutral–neutral reaction | H₂ + OH → H₂O + H |

</details>

## Exercise (a.2) - Reaction constants
Find a gas-phase reaction in UCLCHEM that has a nonzero alpha, beta and gamma, Tmin and Tmax. Compute it's rate at 100, 300 and 500K.

In [ ]:
import numpy as np


def compute_rate(temperature):
    # Implement your function here! Python power is **
    return 0.0


for temperature in [100.0, 300.0, 500.0]:
    rate = compute_rate(temperature)
    print(f"At {temperature} K, the rate is {rate}")

<details>
Taking reaction H3+ + O -> H2O+ -> H from line 2461:
H3+,O,NAN,H2O+,H,NAN,NAN,2.08e-10,-0.4,4.86,10.0,1000.0,0.0,False,0.0
We implement the KA reaction: k = α (T/300)^β exp(-γ/T)

```python
def compute_rate(temperature):
    # H3+,O,NAN,H2O+,H,NAN,NAN,2.08e-10,-0.4,4.86,10.0,1000.0,0.0,False,0.0
    return 2.08e-10 * (temperature / 300.0) ** -(0.4) * np.exp(-4.86 / temperature)


for temperature in [100.0, 300.0, 500.0]:
    rate = compute_rate(temperature)
    print(f"At {temperature} K, the rate is {rate}")
```
</details>

## Accessing UCLCHEM Network directly with Python (Optional)

In [ ]:
import random
import numpy as np
from uclchem.makerates.network import Network

# ── Load the compiled network ──────────────────────────────────────────────
network = Network.from_csv()

# ── Inspect species ────────────────────────────────────────────────────────
species_list = network.get_species_list()
print(f"Total species : {len(species_list)}")

# Species objects carry physical properties
for sp in species_list[:3]:
    print(
        f"  {sp.get_name():10s}  mass={sp.get_mass():5.1f} amu  "
        f"binding energy={sp.get_binding_energy():6.0f} K"
    )
print("  ...")

# ── Inspect reactions ──────────────────────────────────────────────────────
reaction_list = network.get_reaction_list()
print(f"\nTotal reactions : {len(reaction_list)}")

# Count reactions by type
from collections import Counter

type_counts = Counter(r.get_reaction_type() for r in reaction_list)
print("Reaction type counts:")
for rtype, count in sorted(type_counts.items(), key=lambda x: -x[1]):
    print(f"  {rtype:12s} : {count}")

# ── Inspect a few two-body gas-phase reactions ─────────────────────────────
twobody = network.get_reactions_by_types("TWOBODY")
print(f"\nFirst 5 TWOBODY reactions:")
for r in twobody[:5]:
    reac = " + ".join(r.get_pure_reactants())
    prod = " + ".join(r.get_pure_products())
    print(f"  {reac:30s} → {prod}")

# ── Exercise (a.2): rate at 100, 300, 500 K ───────────────────────────────
# Pick a random two-body reaction and compute its rate constant
rxn = random.choice(twobody)
reac = " + ".join(rxn.get_pure_reactants())
prod = " + ".join(rxn.get_pure_products())

print(f"\nRandom TWOBODY reaction:")
print(f"  {reac} → {prod}")
print(f"  α={rxn.get_alpha():.3e}  β={rxn.get_beta():.2f}  γ={rxn.get_gamma():.2f}")
print(f"  T range: {rxn.get_templow():.0f} – {rxn.get_temphigh():.0f} K")
print()


# Modified Arrhenius (Kooij): k = α (T/300)^β exp(−γ/T)
def compute_rate(reaction, temperature):
    return (
        reaction.get_alpha()
        * (temperature / 300.0) ** reaction.get_beta()
        * np.exp(-reaction.get_gamma() / temperature)
    )


for T in [100.0, 300.0, 500.0]:
    print(f"  k({T:.0f} K) = {compute_rate(rxn, T):.4e} cm³ s⁻¹")


## 2  Running the Code

The cell below sets up the fiducial parameters and runs the model.  
Run it first — element conservation values should all be **0 % or < 1 %**.

In [ ]:
import matplotlib.pyplot as plt
import uclchem

# ── Fiducial parameters ────────────────────────────────────────────────────
out_species = ["O2", "CO", "H2O", "HCO+"]

FIDUCIAL = {
    "endAtFinalDensity": False,  # stop at finalTime, not final density
    "freefall": False,  # keep density constant
    "initialDens": 1e4,  # cm⁻³
    "initialTemp": 10.0,  # K
    "finalTime": 6.5e7,  # years
    "rout": 0.1,  # cloud radius / pc
    "points": 1,
    "freezeFactor": 1.0,  # multiplier on freeze-out rate
    "desorb": True,  # non-thermal desorption on/off
    "zeta": 1.0,  # cosmic-ray rate × 1.3×10⁻¹⁷ s⁻¹
    "fo": 3.34e-4,  # elemental O / total H
    "fc": 1.77e-4,  # elemental C / total H
    "fn": 6.18e-5,  # elemental N / total H
    "fhe": 0.1,  # elemental He / total H
    "baseAv": 10.0,  # visual extinction at cloud edge
}

cloud_fiducial = uclchem.model.Cloud(param_dict=FIDUCIAL, out_species=out_species)
cloud_fiducial.check_error()

df_fiducial = cloud_fiducial.get_dataframes()
print("Percentage change in total abundances (should all be ≈ 0 %):")
cloud_fiducial.check_conservation(element_list=["H", "N", "C", "O", "S"])

## 3  Plotting Helper

The functions below are used throughout the exercises.  
- `run_model(params, species)` — merges `params` over the fiducial dict, runs a `Cloud`, and returns a DataFrame.
- `plot_species(df, species_list, ...)` — plots abundance vs. time for the given species.
  - Gas-phase: e.g. `"H2O"`, grain-surface ice: `"#CO"`.
  - Set `obs_limits=True` to overlay the observed upper limits for O₂ and H₂O.

In [ ]:
def run_model(params: dict, species=None) -> object:
    """Run UCLCHEM and return the output DataFrame."""
    species = species or out_species
    p = {**FIDUCIAL, **params}
    cloud = uclchem.model.Cloud(param_dict=p, out_species=species)
    cloud.check_error()
    return cloud.get_dataframes()


def plot_species(
    df, species_list, *, ax=None, title=None, obs_limits=False, xlim=None, ylim=None
):
    """
    Plot abundance vs. time for the requested species.

    Parameters
    ----------
    df           : DataFrame returned by cloud.get_dataframes()
    species_list : list of column names to plot (e.g. ['CO', '#CO', 'HCO+'])
    ax           : optional existing Axes; creates a new figure if None
    title        : axes title string
    obs_limits   : if True, draw observed upper limits for O2 and H2O
    xlim, ylim   : optional axis limits as (lo, hi) tuples
    """
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(7, 4.5))

    for sp in species_list:
        if sp not in df.columns:
            print(f"  Warning: '{sp}' not in output – was it in out_species?")
            continue
        ax.plot(df["Time"], df[sp], label=sp)

    if obs_limits:
        ax.axhline(3e-9, color="black", lw=1.5, ls="--", label="O₂ upper limit")
        ax.axhline(1e-9, color="black", lw=1.5, ls=":", label="H₂O upper limit")

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Time  (years)")
    ax.set_ylabel(r"Abundance  $n(X)/n_\mathrm{H}$")
    if xlim:
        ax.set_xlim(xlim)
    if ylim:
        ax.set_ylim(ylim)
    ax.legend(fontsize=8)
    if title:
        ax.set_title(title)

    if standalone:
        plt.tight_layout()
        plt.show()

---
## Exercise (b) — Gas-Phase H₂O Abundance

Plot H₂O abundance vs. time and explain the stagnation around 10⁵ years.

In [ ]:
plot_species(df_fiducial, ["H2O"], title="Gas-phase H₂O (fiducial)")

### ✏️ Your answer (b)

*Why is there a stagnation in gas-phase water abundance at around 10⁵ years?*

*(Double-click to edit)*

<details>
<summary><b>Solution (b) — click to expand</b></summary>

Gas-phase H₂O decreases stagnates it **freezes out onto grain surfaces**.  
Around 10⁵ years the freeze-out timescale becomes shorter than the chemical production timescale,  
so ice accumulation matches the gas-phase reservoir formation.

</details>

---
## Exercise (c) — Comparison with Observed Upper Limits

Observed upper limits in starless cores / cold protostellar envelopes:

| Species | Upper limit |
|---------|-------------|
| O₂ | $n(\text{O}_2)/n_H < 3 \times 10^{-9}$ |
| H₂O | $n(\text{H}_2\text{O})/n_H < 1 \times 10^{-9}$ |

Plot O₂ and H₂O with and without the upper limit lines.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
plot_species(df_fiducial, ["O2", "H2O"], ax=axes[0], title="Fiducial model")
plot_species(
    df_fiducial,
    ["O2", "H2O"],
    obs_limits=True,
    ax=axes[1],
    title="Fiducial model + observed upper limits",
)
plt.tight_layout()
plt.show()

### ✏️ Your answer (c)

*How do the model predictions compare to the observed upper limits?*

*(Double-click to edit)*

<details>
<summary><b>Solution (c) — click to expand</b></summary>

Both O₂ and H₂O model abundances are **above** the observed upper limits for most of the  
simulation, meaning the fiducial model over-predicts these species relative to what is seen  
in cold, dense interstellar cores.

</details>

---
## Exercise (d) — Varying the Initial Oxygen Abundance

Can you lower the predicted abundances to match the upper limits at 10⁶ years  
by reducing `"fo"` (elemental O / total H)?

> *Hint 1: Decrease the oxygen abundance.*  
> *Hint 2: The parameter to change is `"fo"` in `param_dict`.*

In [ ]:
# ── Modify fo here ────────────────────────────────────────────────────────
fo_test = 7e-7  # adjust this value

df_d = run_model({"fo": fo_test})

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
plot_species(
    df_fiducial,
    ["O2", "H2O"],
    obs_limits=True,
    ax=axes[0],
    title=f"Fiducial  (fo = {FIDUCIAL['fo']:.2e})",
)
plot_species(
    df_d,
    ["O2", "H2O"],
    obs_limits=True,
    ax=axes[1],
    title=f"Reduced oxygen  (fo = {fo_test:.1e})",
)
plt.tight_layout()
plt.show()

### ✏️ Your answer (d)

*What value of `fo` is needed? Is it physically reasonable?*

*(Double-click to edit)*

<details>
<summary><b>Solution (d) — click to expand</b></summary>

Setting **`fo = 7e-7`** brings both O₂ and H₂O below the observed upper limits after 10⁶ years.  
However, this is many orders of magnitude below the cosmic oxygen abundance (~3 × 10⁻⁴),  
so this alone is not a physically convincing explanation.

</details>

---
## Exercise (e) — Increasing the Freeze-Out Factor

Reset `fo` to `3.34e-4`. Shorten `finalTime` to 10⁶ years and increase `freezeFactor`.  
Can you match the upper limits at 10⁶ years?

> *Note: `freezeFactor` scales the rate at which gas-phase species freeze onto grains.  
> A larger value means faster freeze-out.*

In [ ]:
# ── Modify freezeFactor here ──────────────────────────────────────────────
ff_test = 50  # adjust this value

df_e = run_model({"freezeFactor": ff_test, "finalTime": 1e6})

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
plot_species(
    df_fiducial,
    ["O2", "H2O"],
    obs_limits=True,
    ax=axes[0],
    title="Fiducial  (freezeFactor = 1, 10⁷ yr)",
    xlim=(1e2, 1e7),
)
plot_species(
    df_e,
    ["O2", "H2O"],
    obs_limits=True,
    ax=axes[1],
    title=f"freezeFactor = {ff_test}, finalTime = 10⁶ yr",
    xlim=(1e2, 1e6),
)
plt.tight_layout()
plt.show()

### ✏️ Your answer (e)

*What value of `freezeFactor` is needed? Why does a higher freeze-out rate help?*

*(Double-click to edit)*

<details>
<summary><b>Solution (e) — click to expand</b></summary>

Setting **`freezeFactor = 50`** brings the model into agreement with the upper limits at 10⁶ years.  
A higher freeze-out rate locks gas-phase O₂ and H₂O onto grain surfaces faster,  
depleting the gas-phase abundances before they exceed the observed limits.

</details>

---
## Exercise (f) — Turning Off Non-Thermal Desorption

Reset `freezeFactor = 1` and `finalTime = 1e7`.  
Set `desorb = False` to turn off all non-thermal desorption.  
Do the models now agree with observations?

In [ ]:
df_f = run_model({"desorb": False})

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
plot_species(
    df_fiducial,
    ["O2", "H2O"],
    obs_limits=True,
    ax=axes[0],
    title="Fiducial  (desorb = True)",
)
plot_species(df_f, ["O2", "H2O"], obs_limits=True, ax=axes[1], title="desorb = False")
plt.tight_layout()
plt.show()

### ✏️ Your answer (f)

*Does turning off desorption help? What does this tell you about the modelled desorption processes?*

*(Double-click to edit)*

<details>
<summary><b>Solution (f) — click to expand</b></summary>

Yes — with `desorb = False` both species fall below the observed upper limits.  
This implies that the **non-thermal desorption efficiencies** (e.g. photodesorption) assumed  
in the standard model are likely **overestimated**, returning too many molecules to the gas phase.

</details>

---
## Exercise (g) — CO and HCO⁺: Chemical Link

Restore all parameters to their fiducial values.  
Plot the CO and HCO⁺ gas-phase abundances and explain their relationship.

In [ ]:
out_species_ghijk = ["CO", "H2O", "O2", "HCO+", "#CO"]
df_base = run_model({}, species=out_species_ghijk)

plot_species(df_base, ["CO", "HCO+"], title="CO and HCO⁺ — fiducial")

### ✏️ Your answer (g)

*Explain the relationship between CO and HCO⁺ abundances in terms of the chemistry linking them.*

*(Double-click to edit)*

<details>
<summary><b>Solution (g) — click to expand</b></summary>

HCO⁺ is primarily formed by:

$$\mathrm{CO + H_3^+ \rightarrow HCO^+ + H_2}$$

CO is therefore a **necessary precursor**.  The HCO⁺ abundance rises and falls in step with CO,  
but with a slight lag because HCO⁺ needs time to build up after CO becomes available.

</details>

---
## Exercise (h) — Effect of Higher Density on CO and HCO⁺

Increase `initialDens` by two orders of magnitude to 10⁶ cm⁻³.  
Explain the effect on CO. Why is there a steep drop? How does this affect HCO⁺?

In [ ]:
df_highdens = run_model({"initialDens": 1e6}, species=out_species_ghijk)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
plot_species(df_base, ["CO", "HCO+"], ax=axes[0], title="Fiducial  (n = 10⁴ cm⁻³)")
plot_species(
    df_highdens, ["CO", "HCO+"], ax=axes[1], title="High density  (n = 10⁶ cm⁻³)"
)
plt.tight_layout()
plt.show()

### ✏️ Your answer (h)

*Explain the effect of higher density on CO. Why the steep drop? How does this affect HCO⁺?*

*(Double-click to edit)*

<details>
<summary><b>Solution (h) — click to expand</b></summary>

At higher density:
- CO is initially more abundant (more material at the same temperature).
- A **steep drop appears around 10⁴ years** due to **CO freeze-out**: the freeze-out timescale  
  scales as 1/n, so at 100× higher density freeze-out is 100× faster.
- The CO depletion directly reduces HCO⁺, since HCO⁺ formation requires CO as a reactant.

</details>

---
## Exercise (i) — CO on the Grains

Verify your freeze-out interpretation by plotting grain-surface CO (`#CO`)  
alongside gas-phase CO for both the fiducial and high-density runs.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
plot_species(
    df_base, ["CO", "#CO", "$CO"], ax=axes[0], title="CO gas vs. ice — fiducial"
)
plot_species(
    df_highdens, ["CO", "#CO", "$CO"], ax=axes[1], title="CO gas vs. ice — high density"
)
plt.tight_layout()
plt.show()

### ✏️ Your answer (i)

*What happens to grain-surface CO when gas-phase CO drops? Does this confirm your answer in (h)?*

*(Double-click to edit)*

<details>
<summary><b>Solution (i) — click to expand</b></summary>

The **grain-surface (ice) CO abundance rises** exactly when the gas-phase CO drops,  
confirming that the gas-phase depletion is driven by freeze-out onto dust grains.

</details>

---
## Exercise (j) — Effect of Temperature on CO Ice

Keep `initialDens = 1e6` from (h)/(i) and raise the temperature to **30 K**.  
Explain the effect on the CO ice abundance.

Warning: this cell might take a lot longer to run than the other ones due to numerical stiffness!

In [ ]:
df_highT = run_model(
    {
        "initialDens": 1e6,
        "initialTemp": 30.0,
        "finalTime": 1e6,
        # "reltol": 1e-6,
        # "abstol": 1e-9,
        "abstol_ice_factor": 1e-12,
        "abstol_ice_min": 1e-22,
    },
    species=out_species_ghijk,
)

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
plot_species(df_base, ["CO", "#CO", "@CO"], ax=axes[0], title="Fiducial  (10 K, n=10⁴)")
plot_species(
    df_highdens, ["CO", "#CO", "@CO"], ax=axes[1], title="High density  (10 K, n=10⁶)"
)
plot_species(
    df_highT,
    ["CO", "#CO", "@CO"],
    ax=axes[2],
    title="High density + high T  (30 K, n=10⁶)",
)
plt.tight_layout()
plt.show()

### ✏️ Your answer (j)

*Explain the effect of raising the temperature to 30 K on the CO ice abundance.*

*(Double-click to edit)*

<details>
<summary><b>Solution (j) — click to expand</b></summary>

At 30 K **thermal desorption** becomes efficient enough to return CO from ice back to the gas.  
Even at the high density where freeze-out dominates at 10 K, the gas-phase CO abundance  
remains *higher* than the ice abundance at 30 K — meaning no lasting freeze-out occurs.

</details>

---
## Exercise (k) — Effect of Cosmic Ray Ionisation Rate on HCO⁺

Restore all parameters to fiducial values.  
Increase `zeta` (the cosmic-ray ionisation factor) to **10**.  
What effect does this have on the HCO⁺ abundance, and why?

In [ ]:
df_zeta = run_model({"zeta": 10.0}, species=out_species_ghijk)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
plot_species(df_base, ["CO", "HCO+"], ax=axes[0], title="Fiducial  (ζ = 1)")
plot_species(
    df_zeta, ["CO", "HCO+"], ax=axes[1], title="High cosmic-ray rate  (ζ = 10)"
)
plt.tight_layout()
plt.show()


### ✏️ Your answer (k)

*What is the effect of ζ = 10 on HCO⁺? By roughly what factor does it change, and why?*

*(Double-click to edit)*

<details>
<summary><b>Solution (k) — click to expand</b></summary>

A 10× higher cosmic-ray ionisation rate raises the HCO⁺ abundance. 

The reasoning:

1. Cosmic rays ionise H₂, ultimately producing H₃⁺.  
2. H₃⁺ reacts with CO to form HCO⁺:  
   $$\mathrm{CO + H_3^+} \rightarrow \mathrm{HCO^+ + H}_2$$  
3. Formation of H₃⁺ scales linearly with ζ; destruction of HCO⁺ (dissociative recombination  
   with electrons) scales with the electron density, which also scales with $\sqrt{\zeta}$.  
4. The steady-state HCO⁺ abundance therefore scales as **$\sqrt{\zeta}$**.

</details>